# bob, explained - Episode 12: What is ours: bob against OpenFPGA and Aegis

What is ours: bob against OpenFPGA and Aegis

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep12Novelty
# =============================================================================
#  bob, explained - EPISODE 12: What is ours: bob against OpenFPGA and Aegis
#  What is ours: bob against OpenFPGA and Aegis
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E12S1Lineage
#      E12S2Openfpga
#      E12S3Aegis
#      E12S4Ours
#      E12S5Notours
#      E12S6Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 12 - What is ours: bob against OpenFPGA, Aegis, ZUMA, prjxray
# =============================================================================

def s1_lineage(sc):
    sc.heading("Almost nothing here is invented",
               "and that is the point - every part is grounded in something already tested")

    rows = [
        ("CLB", "AMD UG474", "LUT6_2 fracture, MUXCY/XORCY, FDRE/FDSE, routable CE/SR", C_RTL),
        ("BRAM", "AMD UG473", "RAMB18E1 behaviour, write modes, DOx_REG", C_RTL),
        ("DSP", "AMD UG479", "A25/B18/D25, pre-adder, 25x18, P48, cascade", C_RTL),
        ("configuration", "AMD UG470", "packets, FAR, CMD, STAT, CRC, startup, AGHIGH", C_BIT),
        ("CRC", "prjxray crc.py", "CRC-32C over {register, data}", C_BIT),
        ("routing", "OpenFPGA k6_frac_N10", "tileable, L4, Wilton Fs 3, fc 0.15 / 0.10", C_VPR),
        ("chain", "OpenFPGA scan_chain, Aegis", "shift register plus a shadow register", C_GRF),
        ("PnR", "Betz & Rose; McMurchie & Ebeling", "annealing placer, PathFinder router", C_PY),
        ("FASM", "F4PGA", "feature = value between PnR and bits", C_PY),
        ("area idea", "ZUMA (FCCM 2012)", "configuration memory in host LUTRAM - considered", DIM),
    ]
    g = VGroup()
    for a, b, c, col in rows:
        g.add(VGroup(mono(a, 18, col), mono(b, 17, INK), Text(c, font_size=15, color=DIM))
              .arrange(RIGHT, buff=0.35, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 2.2)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 6.0)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.2)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.1),
            run_time=2.6)

    note = Text("Where bob diverges, PLAN.md and the report say so explicitly - "
                "there is a whole section listing it.", font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(note), run_time=0.8)
    sc.wait(2.2)


def s2_openfpga(sc):
    sc.heading("bob against OpenFPGA", "the project bob borrows its method from - and it is far bigger")

    a = Text("OpenFPGA", font_size=30, color=C_VPR, weight="BOLD").move_to(np.array([-3.4, 2.1, 0]))
    b = Text("bob", font_size=30, color=C_RTL, weight="BOLD").move_to(np.array([3.4, 2.1, 0]))
    sc.play(FadeIn(a), FadeIn(b), run_time=0.5)

    rows = [
        ("many architectures, from one XML", "one architecture, one board"),
        ("Verilog AND SPICE, ASIC-oriented", "Verilog only, an overlay on a real FPGA"),
        ("area and power models", "none"),
        ("scan_chain, frame_based, memory_bank...", "scan chain AND UG470 frames, on ONE memory"),
        ("a mature, general framework", "one device description drives RTL, VPR arch,"),
        ("", "models, FASM map and host tools"),
        ("verified mostly in simulation", "every milestone proven on real hardware"),
    ]
    g = VGroup()
    for l, r in rows:
        lt = Text(l, font_size=17, color=DIM)
        rt = Text(r, font_size=17, color=INK)
        g.add(VGroup(lt, rt).arrange(RIGHT, buff=0.6, aligned_edge=UP))
    for row in g:
        row[1].align_to(g[0][1], LEFT).shift(RIGHT * 5.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(VGroup(a, b), DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=UP * 0.1) for r in g], lag_ratio=0.15),
            run_time=2.2)

    honest = Text("OpenFPGA is a research framework with years of work behind it. "
                  "bob is one fabric on one board. The comparison is about METHOD, not scale.",
                  font_size=19, color=C_BIT)
    honest.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(honest), run_time=0.9)
    sc.wait(2.2)


def s3_aegis(sc):
    sc.heading("bob against Aegis", "a full-stack open FPGA going to real silicon - a different goal entirely")

    rows = [
        ("target", "GF180MCU / Sky130 silicon", "the PL of an XC7Z020"),
        ("written in", "Dart / ROHD -> SystemVerilog", "Verilog, generated from Python"),
        ("logic", "LUT4, one per tile", "LUT6 fracturable + carry + FF"),
        ("routing", "per-tile switchbox, 1-4 tracks per edge", "VPR rr graph, L4, W = 24, Wilton"),
        ("configuration", "one scan chain, shift + shadow, cfgLoad", "chain AND UG470 frames"),
        ("integrity", "none in the chain", "CRC-32C, length, IDCODE, per-write CRC"),
        ("partial reconfiguration", "no", "yes, with state provably kept"),
        ("what bob took", "the shift + shadow idea, the I/O and clock notes", ""),
    ]
    g = VGroup()
    for a, b, c in rows:
        g.add(VGroup(mono(a, 17, C_BIT), Text(b, font_size=16, color=DIM),
                     Text(c, font_size=16, color=INK))
              .arrange(RIGHT, buff=0.35, aligned_edge=UP))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 2.9)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 8.0)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.15),
            run_time=2.4)

    note = Text("Aegis solves a harder problem than bob does: it has to be manufacturable. "
                "bob only has to be correct, and provable, on one board.",
                font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.2)


def s4_ours(sc):
    sc.heading("What is actually ours", "six things that are not borrowed from anywhere")

    items = [
        ("Hardware proof at every step",
         "sixteen milestones, each ending on the PYNQ-Z2. Simulation never closed one."),
        ("One description, five consumers",
         "device.py generates the RTL, the VPR architecture, the models, the FASM map"),
        ("", "and the host tools. They cannot drift, and pytest proves they have not."),
        ("Two configuration paths on ONE memory",
         "a UG470 frame engine and a scan chain, writing the same bits, both kept working"),
        ("Partial reconfiguration that keeps state provably",
         "AGHIGH holds the user clock; every register is gated by gce, so state is exact"),
        ("Golden co-simulation",
         "source Verilog and a golden netlist run beside the whole FPGA loaded from a real .bit"),
        ("A stand-in board",
         "every hardware check is tested - passing AND failing - before the board sees it"),
    ]
    g = VGroup()
    for a, b in items:
        if a:
            g.add(VGroup(mono("*", 20, C_BIT), Text(a, font_size=20, color=INK),)
                  .arrange(RIGHT, buff=0.3, aligned_edge=DOWN))
            g.add(Text(b, font_size=16, color=DIM))
        else:
            g.add(Text(b, font_size=16, color=DIM))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.16)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.12),
            run_time=2.8)
    sc.wait(2.4)


def s5_notours(sc):
    sc.heading("And what bob is not", "saying this plainly is part of the work")

    items = [
        "one small fabric, on one board, with one BLE per CLB",
        "no timing-driven place and route - VPR's delays are the reference 40 nm numbers,",
        "not the emulated fabric's",
        "no ASIC flow, no SPICE, no area or power models",
        "one clock domain for guest designs; no async resets or latches",
        "K and W are fixed per build - changing them means a full regenerate and rebuild",
        "far less coverage of architectures and devices than OpenFPGA",
        "partial reconfiguration has no region protection: the host decides which frames",
    ]
    g = VGroup(*[VGroup(mono("-", 20, C_ERR), Text(t, font_size=19, color=INK))
                 .arrange(RIGHT, buff=0.3, aligned_edge=DOWN) for t in items])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.15),
            run_time=2.4)

    last = Text("A claim you cannot fail is not a claim. Every limit here is written down "
                "in PLAN.md and in the report.", font_size=20, color=C_BIT)
    last.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(last), run_time=0.9)
    sc.wait(2.4)


def s6_files(sc):
    sc.files_used(
        inputs=[("PLAN.md", "section 9: references, and what each is used for"),
                ("docs/project/REPORT.md", "section 20: follows vs diverges"),
                ("docs/project/GUIDE.md", "section 5: how bob compares")],
        generated=[("project.html", "the interactive report"),
                   ("guide.html", "the learning guide"),
                   ("arch.html", "the interactive die slice")],
        verified=[("tests/test_vpr.py", "the reference routing parameters are still kept"),
                  ("tools/bob/arch/*.stamp", "which OpenFPGA image built the graph"),
                  ("docs/hwtest/results.log", "the hardware claim, run by run")])


EP12 = [s1_lineage, s2_openfpga, s3_aegis, s4_ours, s5_notours, s6_files]


class Ep12Novelty(BobScene):
    def construct(self):
        self.titlecard("EPISODE 12", "What is ours",
                       "bob against OpenFPGA, Aegis and ZUMA - honestly")
        for i, part in enumerate(EP12):
            part(self)
            if i < len(EP12) - 1:
                clear_all(self)


class E12S1Lineage(BobScene):
    def construct(self): s1_lineage(self)


class E12S2Openfpga(BobScene):
    def construct(self): s2_openfpga(self)


class E12S3Aegis(BobScene):
    def construct(self): s3_aegis(self)


class E12S4Ours(BobScene):
    def construct(self): s4_ours(self)


class E12S5Notours(BobScene):
    def construct(self): s5_notours(self)


class E12S6Files(BobScene):
    def construct(self): s6_files(self)